In [0]:
# Smart Data Deduplication Pipeline
# Notebook 2: Data Ingestion

DATABASE_NAME = "smart_data_deduplication"

print("Using database:", DATABASE_NAME)

Using database: smart_data_deduplication


In [0]:
INPUT_PATH = "//Volumes/workspace/data/vishnu/sales_dataset.csv"

SOURCE_NAME = "base_source"
RECORD_ID_COLUMN = None

print("Input path:", INPUT_PATH)
print("Source:", SOURCE_NAME)

Input path: //Volumes/workspace/data/vishnu/sales_dataset.csv
Source: base_source


In [0]:
from pyspark.sql.functions import col

input_path = "/" + INPUT_PATH.lstrip("/")

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(input_path)
)

print("Records:", df.count())
print("Columns:", df.columns)

display(df)

Records: 1500
Columns: ['entity_type', 'record_id', 'name', 'category', 'description', 'value', 'source', 'updated_at', 'test_case']


entity_type,record_id,name,category,description,value,source,updated_at,test_case
sale,SALE01108,Runing Shoes Prism Series 57,Fashion,Running Shoes order; quantity=5; unit_price=5500 INR,27500,source_C,2026-09-03 17:40:00,NEGATIVE_NEAR_DUPLICATE
sale,SALE00831,Mouse Vertex Series 56,Accessories,Mouse order; quantity=2; unit_price=350 INR,700,source_E,2026-09-03 16:24:00,POSITIVE_CLEAN
sale,SALE00862,Mixer Grinder Aster Series 58,Home Appliances,Mixer Grinder order; quantity=3; unit_price=3000 INR,9000,source_A,2026-09-03 18:28:00,POSITIVE_CLEAN
sale,SALE00420,Badminton Racket Luna Series 28,Sports,Badminton Racket order; quantity=4; unit_price=6450 INR,25800,source_G,2026-09-02 13:00:00,POSITIVE_CLEAN
sale,SALE00736,Smartphone Titan Series 50,Electronics,Smartphone order; quantity=3; unit_price=17750 INR,53250,source_A,2026-09-03 10:04:00,POSITIVE_CLEAN
sale,SALE00967,Mixer Grinder Apex Series 65,Home Appliances,Mixer Grinder order; quantity=3; unit_price=10700 INR,32100,source_A,2026-09-04 01:28:00,POSITIVE_CLEAN
sale,SALE00396,Mouse Apex Series 27,Accessories,Mouse order; quantity=1; unit_price=4750 INR,4750,source_D,2026-09-02 11:24:00,POSITIVE_CLEAN
sale,SALE00190,Running Shoes Vertex Series 13,Fashion,Running Shoes order; quantity=2; unit_price=5650 INR,11300,source_A,2026-09-01 21:40:00,POSITIVE_CLEAN
sale,SALE00007,Mixer Grinder Vega Series 1,Home Appliances,Mixer Grinder order; quantity=5; unit_price=5050 INR,25250,source_G,2026-09-01 09:28:00,POSITIVE_CLEAN
sale,SALE00372,Coffee Pack Orion Series 25,Grocery,Coffee Pack order; quantity=4; unit_price=750 INR,3000,source_A,2026-09-02 09:48:00,POSITIVE_CLEAN


In [0]:
# Check record IDs

if RECORD_ID_COLUMN is None:
    RECORD_ID_COLUMN = "record_id"

missing_id_count = df.filter(
    col(RECORD_ID_COLUMN).isNull()
).count()

duplicate_id_count = (
    df.groupBy(RECORD_ID_COLUMN)
      .count()
      .filter(
          (col("count") > 1) &
          col(RECORD_ID_COLUMN).isNotNull()
      )
      .count()
)

print("Record ID column:", RECORD_ID_COLUMN)
print("Missing IDs:", missing_id_count)
print("Duplicate IDs:", duplicate_id_count)

Record ID column: record_id
Missing IDs: 30
Duplicate IDs: 150


In [0]:
from pyspark.sql.functions import col, struct, to_json, lit, current_timestamp

# Keep all original columns except record_id inside record_data
json_columns = [
    c for c in df.columns
    if c != RECORD_ID_COLUMN
]

staging_df = (
    df
    .withColumn(
        "record_data",
        to_json(struct(*[col(c) for c in json_columns]))
    )
    .withColumn(
        "source",
        lit(SOURCE_NAME)
    )
    .withColumn(
        "loaded_at",
        current_timestamp()
    )
    .select(
        col(RECORD_ID_COLUMN).cast("string").alias("record_id"),
        col("record_data"),
        col("source"),
        col("loaded_at")
    )
)

display(staging_df)

record_id,record_data,source,loaded_at
SALE01108,"{""entity_type"":""sale"",""name"":""Runing Shoes Prism Series 57"",""category"":""Fashion"",""description"":""Running Shoes order; quantity=5; unit_price=5500 INR"",""value"":27500,""source"":""source_C"",""updated_at"":""2026-09-03 17:40:00"",""test_case"":""NEGATIVE_NEAR_DUPLICATE""}",base_source,2026-11-24T16:18:15.030Z
SALE00831,"{""entity_type"":""sale"",""name"":""Mouse Vertex Series 56"",""category"":""Accessories"",""description"":""Mouse order; quantity=2; unit_price=350 INR"",""value"":700,""source"":""source_E"",""updated_at"":""2026-09-03 16:24:00"",""test_case"":""POSITIVE_CLEAN""}",base_source,2026-11-24T16:18:15.030Z
SALE00862,"{""entity_type"":""sale"",""name"":""Mixer Grinder Aster Series 58"",""category"":""Home Appliances"",""description"":""Mixer Grinder order; quantity=3; unit_price=3000 INR"",""value"":9000,""source"":""source_A"",""updated_at"":""2026-09-03 18:28:00"",""test_case"":""POSITIVE_CLEAN""}",base_source,2026-11-24T16:18:15.030Z
SALE00420,"{""entity_type"":""sale"",""name"":""Badminton Racket Luna Series 28"",""category"":""Sports"",""description"":""Badminton Racket order; quantity=4; unit_price=6450 INR"",""value"":25800,""source"":""source_G"",""updated_at"":""2026-09-02 13:00:00"",""test_case"":""POSITIVE_CLEAN""}",base_source,2026-11-24T16:18:15.030Z
SALE00736,"{""entity_type"":""sale"",""name"":""Smartphone Titan Series 50"",""category"":""Electronics"",""description"":""Smartphone order; quantity=3; unit_price=17750 INR"",""value"":53250,""source"":""source_A"",""updated_at"":""2026-09-03 10:04:00"",""test_case"":""POSITIVE_CLEAN""}",base_source,2026-11-24T16:18:15.030Z
SALE00967,"{""entity_type"":""sale"",""name"":""Mixer Grinder Apex Series 65"",""category"":""Home Appliances"",""description"":""Mixer Grinder order; quantity=3; unit_price=10700 INR"",""value"":32100,""source"":""source_A"",""updated_at"":""2026-09-04 01:28:00"",""test_case"":""POSITIVE_CLEAN""}",base_source,2026-11-24T16:18:15.030Z
SALE00396,"{""entity_type"":""sale"",""name"":""Mouse Apex Series 27"",""category"":""Accessories"",""description"":""Mouse order; quantity=1; unit_price=4750 INR"",""value"":4750,""source"":""source_D"",""updated_at"":""2026-09-02 11:24:00"",""test_case"":""POSITIVE_CLEAN""}",base_source,2026-11-24T16:18:15.030Z
SALE00190,"{""entity_type"":""sale"",""name"":""Running Shoes Vertex Series 13"",""category"":""Fashion"",""description"":""Running Shoes order; quantity=2; unit_price=5650 INR"",""value"":11300,""source"":""source_A"",""updated_at"":""2026-09-01 21:40:00"",""test_case"":""POSITIVE_CLEAN""}",base_source,2026-11-24T16:18:15.030Z
SALE00007,"{""entity_type"":""sale"",""name"":""Mixer Grinder Vega Series 1"",""category"":""Home Appliances"",""description"":""Mixer Grinder order; quantity=5; unit_price=5050 INR"",""value"":25250,""source"":""source_G"",""updated_at"":""2026-09-01 09:28:00"",""test_case"":""POSITIVE_CLEAN""}",base_source,2026-11-24T16:18:15.030Z
SALE00372,"{""entity_type"":""sale"",""name"":""Coffee Pack Orion Series 25"",""category"":""Grocery"",""description"":""Coffee Pack order; quantity=4; unit_price=750 INR"",""value"":3000,""source"":""source_A"",""updated_at"":""2026-09-02 09:48:00"",""test_case"":""POSITIVE_CLEAN""}",base_source,2026-11-24T16:18:15.030Z


In [0]:
# Clear previous staging data for this test run

spark.sql(f"""
TRUNCATE TABLE {DATABASE_NAME}.staging_records
""")

print("Previous staging data cleared.")

Previous staging data cleared.


In [0]:
# Write incoming records to Delta staging table

staging_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable(f"{DATABASE_NAME}.staging_records")

print("Data successfully written to staging_records!")

Data successfully written to staging_records!


In [0]:
staging_check = spark.sql(f"""
SELECT *
FROM {DATABASE_NAME}.staging_records
""")

print("Total staging records:", staging_check.count())

display(staging_check)